# Taller #1 — Datasets, tokenización y embeddings
### Notebook orquestador (proyecto modularizado `Entrega_2_PNL`)

Este notebook **no contiene la lógica del proyecto**: solo monta Google Drive,
instala dependencias, agrega el proyecto al `sys.path` y llama a los
pipelines definidos en `src/`. Todo el código reutilizable vive en los
módulos `.py` dentro de `src/ejercicio1_tokenizacion`, `src/ejercicio2_embeddings`
y `src/ejercicio3_pdf_embeddings`.

**Antes de ejecutar:** sube toda la carpeta `Entrega_2_PNL` a tu Google Drive
(con la misma estructura que trae), y coloca tus PDFs en
`Entrega_2_PNL/pdfs_taller/` si vas a correr el Ejercicio 3.


## 0. Configuración del entorno (Drive + dependencias + sys.path)

In [1]:
# Solo en Google Colab: montar Drive si está disponible.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print("No se detectó Google Colab; se asumirá ejecución local.")


No se detectó Google Colab; se asumirá ejecución local.


In [2]:
import os
from pathlib import Path

# Elige el entorno de ejecución.
# - 'local': usa la carpeta actual del proyecto o sus carpetas padres.
# - 'drive': intenta usar /content/drive/MyDrive/Entrega_2_PNL.
ENTORNO = "local"


def encontrar_proyecto():
    candidatos = []
    if ENTORNO == "drive":
        candidatos.append("/content/drive/MyDrive/Entrega_2_PNL")

    for base in [os.environ.get("PNL_BASE_DIR"), str(Path.cwd().resolve())]:
        if base:
            candidatos.append(base)

    for base in [Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive/Entrega_2_PNL"), Path("/content")]:
        for candidata in [base, *base.parents]:
            if (candidata / "src").exists() and (candidata / "requirements.txt").exists():
                return str(candidata.resolve())

    for candidata in candidatos:
        if candidata and os.path.exists(candidata) and os.path.exists(os.path.join(candidata, "requirements.txt")):
            return str(Path(candidata).expanduser().resolve())

    return None


RUTA_PROYECTO = encontrar_proyecto()
if not RUTA_PROYECTO:
    raise FileNotFoundError("No se pudo localizar la carpeta del proyecto. Cambia ENTORNO o ajusta la ruta manualmente.")

RUTA_PROYECTO = str(Path(RUTA_PROYECTO).expanduser().resolve())
print("✓ Proyecto encontrado en:", RUTA_PROYECTO)


✓ Proyecto encontrado en: /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL


In [3]:
# Instalar dependencias del proyecto solo si es necesario.
import os
if os.path.exists(os.path.join(RUTA_PROYECTO, 'requirements.txt')):
    !pip install -q -r "$RUTA_PROYECTO/requirements.txt"


In [4]:
# Agregar el proyecto al path de Python para poder hacer `from src... import ...`
import sys
from pathlib import Path

# Añadir también la carpeta padre del proyecto si el paquete no se encuentra.
proyecto_parent = str(Path(RUTA_PROYECTO).resolve())
if proyecto_parent not in sys.path:
    sys.path.insert(0, proyecto_parent)

# Para notebooks ejecutados desde dentro de la carpeta notebooks/, también
# es útil agregar la raíz del proyecto como un nivel superior.
if str(Path(RUTA_PROYECTO).resolve().parent) not in sys.path:
    sys.path.insert(0, str(Path(RUTA_PROYECTO).resolve().parent))

# Apuntar la configuración global del proyecto a la carpeta detectada,
# para que TODOS los resultados se guarden ahí (y persistan entre sesiones).
from src import config
config.BASE_DIR = RUTA_PROYECTO
config.refrescar_rutas()
config.fijar_semillas()

print("BASE_DIR       :", config.BASE_DIR)
print("OUTPUT_DIR_EJ1 :", config.OUTPUT_DIR_EJ1)
print("OUTPUT_DIR_EJ2 :", config.OUTPUT_DIR_EJ2)
print("OUTPUT_DIR_EJ3 :", config.OUTPUT_DIR_EJ3)
print("PDF_DIR        :", config.PDF_DIR)


BASE_DIR       : /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL
OUTPUT_DIR_EJ1 : /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados/ejercicio1_tokenizacion
OUTPUT_DIR_EJ2 : /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados/ejercicio2_embeddings
OUTPUT_DIR_EJ3 : /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados/ejercicio3_pdf_embeddings
PDF_DIR        : /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/pdfs_taller


---
## Ejercicio 1 — Tokenización: WordPiece (BETO) vs. SentencePiece (t5-small)

Cubre las actividades 1 a 7 del enunciado:
1. Cargar Conll2002 y Ancora.
2. Preprocesar train/validation/test de Ancora (split manual 70/15/15).
3. Mostrar ejemplos originales antes de tokenizar.
4. Tokenizar con WordPiece (BETO).
5. Tokenizar con SentencePiece (t5-small).
6. Comparar ambos tokenizadores.
7. Conclusión sobre cuál conserva mejor la estructura léxica del español.


In [5]:
from src.ejercicio1_tokenizacion.pipeline import ejecutar_ejercicio1

resultado_ej1 = ejecutar_ejercicio1(config.OUTPUT_DIR_EJ1, n_ejemplos=3)


/home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



 EJERCICIO 1: TOKENIZACIÓN (WordPiece vs SentencePiece)

Cargando y preprocesando Ancora desde Universal Dependencies...
✓ Ancora unificado: 17662 oraciones totales.
✓ Split manual Ancora -> Train: 12363 (70.0%), Val: 2649 (15.0%), Test: 2650 (15.0%)

Cargando y preprocesando CoNLL2002 vía NLTK...
✓ CoNLL2002 procesado con éxito.

Resumen final de estructuras cargadas:
 - Ancora    -> Train: 12363, Val: 2649, Test: 2650 oraciones.
 - CoNLL2002 -> Train: 8323, Val: 1915, Test: 1517 oraciones.

--- Primeras 3 oraciones originales de Ancora (Train) ---
Oración 1: Las comisiones netas por servicios aumentaron un 6,17 % hasta 3.563 millones , mientras que su resultado por operaciones financieras creció un 99,72 % hasta 703 millones .
Oración 2: También la comunicación puede tener su vehículo en la piel de los animales .
Oración 3: A medida que avanza la temporada de reproducción , la mayoría de los machos quedan atados a su primera nidada y no pueden aceptar más responsabilidades paternale


Cargando tokenizador SentencePiece de t5-small...

Corpus: Ancora (Train)  |  Tokenizador: WordPiece (BETO)

[Oración 1] Original (28 palabras):
  Las | comisiones | netas | por | servicios | aumentaron | un | 6,17 | % | hasta | 3.563 | millones | , | mientras | que | su | resultado | por | operaciones | financieras | creció | un | 99,72 | % | hasta | 703 | millones | .
  Palabras fragmentadas por el vocabulario:
    '6,17' → ['6', ',', '1', '##7']
    '3.563' → ['3', '.', '56', '##3']
    '99,72' → ['99', ',', '7', '##2']
    '703' → ['7', '##03']
  Subtokens obtenidos: ['Las', 'comisiones', 'netas', 'por', 'servicios', 'aumentaron', 'un', '6', ',', '1', '##7', '%', 'hasta', '3', '.', '56', '##3', 'millones', ',', 'mientras', 'que', 'su', 'resultado', 'por', 'operaciones', 'financieras', 'creció', 'un', '99', ',', '7', '##2', '%', 'hasta', '7', '##03', 'millones', '.']
  Resumen: 28 palabras originales  →  38 subtokens.

[Oración 2] Original (14 palabras):
  También | la | comunicaci

In [6]:
# Vista rápida de la tabla comparativa generada
resultado_ej1["df_comparacion"].head(15)


,corpus,oracion_idx,n_palabras,n_tokens_wordpiece,n_tokens_sentencepiece
0,Ancora (Train),0,28,38,66
1,Ancora (Train),1,14,14,33
2,Ancora (Train),2,28,34,63
3,Ancora (Validation),0,49,60,111
4,Ancora (Validation),1,15,17,32
5,Ancora (Validation),2,38,43,98
6,Ancora (Test),0,13,14,33
7,Ancora (Test),1,12,14,32
8,Ancora (Test),2,5,5,8
9,CoNLL2002 (Train),0,11,13,16


---
## Ejercicio 2 — Word2Vec y FastText sobre un corpus en español

Cubre las actividades 1 a 5 del enunciado:
1. Cargar y preprocesar corpus en español con `streaming=True`.
2. Limpiar cada oración (sin puntuación, stopwords, números, tokens vacíos).
3. Entrenar Word2Vec y FastText (`vector_size=300, window=5, min_count=5, workers=4`).
4. Visualizar con PCA y t-SNE para subconjuntos de 100 / 5000 / 10000 palabras.
5. Analizar similitud semántica y palabras OOV (solo posible con FastText).

>  Este paso puede tardar bastante (descarga en streaming + entrenamiento).
> Puedes reducir `N_SENTENCIAS` para pruebas rápidas.


In [7]:
N_SENTENCIAS = 50_000  # Reduce este valor (p. ej. 5_000) para pruebas rápidas

from src.ejercicio2_embeddings.pipeline import ejecutar_ejercicio2

resultado_ej2 = ejecutar_ejercicio2(
    config.OUTPUT_DIR_EJ2,
    n_sentencias=N_SENTENCIAS,
    tamanos_visualizacion=(100, 5000, 10000),
)



 EJERCICIO 2: WORD2VEC Y FASTTEXT SOBRE CORPUS EN ESPAÑOL

Cargando corpus 'allenai/c4' (es) en modo streaming (objetivo: 50000 oraciones)...
Lote 0 guardado: 10000 oraciones
Lote 1 guardado: 10000 oraciones
Lote 2 guardado: 10000 oraciones
Lote 3 guardado: 10000 oraciones
Lote 4 guardado: 10000 oraciones
Total de oraciones preprocesadas: 50000

Ejemplos de oraciones preprocesadas:
['comprar', 'zapatillas', 'niña', 'chancla', 'goma', 'detrás', 'gioseppo', 'rosa', 'online']
['zapatillas', 'niña', 'chancla', 'goma', 'detrás', 'gioseppo', 'rosa']
['zapatillas', 'casa', 'niña', 'otoño', 'invierno', 'zapatillas', 'niña', 'chancla', 'goma', 'detrás', 'gioseppo', 'rosa', 'modelo', 'alice', 'paño', 'suela', 'tela', 'cómodas', 'debido', 'sujección', 'goma', 'viene', 'mochila', 'regalo']

Entrenando Word2Vec...
Word2Vec entrenado. Vocabulario: 24941 palabras. Guardado en /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados/ejercicio2_embeddings/word2vec_es.model

Entrenando FastText..

---
## Ejercicio 3 — PDFs → Chunking → Embeddings (Sentence-Transformers)

Cubre las actividades 1 a 5 del enunciado:
1. Cargar PDFs con LangChain + `pymupdf4llm`.
2. Fragmentar el texto (`RecursiveCharacterTextSplitter`, `chunk_size=1000`, `chunk_overlap=200`).
3. Generar embeddings con 4 modelos de Sentence-Transformers.
4. Consultar similitud semántica (coseno) y recuperar el fragmento más similar.
5. Visualizar en PCA la consulta vs. el fragmento recuperado, por modelo.

> Coloca tus archivos `.pdf` en `Entrega_2_PNL/pdfs_taller/` (en tu Drive)
> antes de ejecutar esta celda.


In [8]:
CONSULTA_EJEMPLO = "¿Cuál es el principio de precaución en materia ambiental?"

from src.ejercicio3_pdf_embeddings.pipeline import ejecutar_ejercicio3

resultado_ej3 = ejecutar_ejercicio3(
    pdf_dir=config.PDF_DIR,
    output_dir=config.OUTPUT_DIR_EJ3,
    consulta=CONSULTA_EJEMPLO,
)



 EJERCICIO 3: PDFs -> CHUNKING -> EMBEDDINGS (Sentence-Transformers)

Se encontraron 1 archivos PDF en '/home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/pdfs_taller'.
  Cargando: /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/pdfs_taller/colaboracion1.pdf
✓ Total de documentos cargados: 1
✓ Documentos fragmentados en 47 chunks (chunk_size=1000, chunk_overlap=200).

Cargando modelo de embeddings: multilingual-e5-base (intfloat/multilingual-e5-base)...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.39s/it]


✓ Embeddings generados para 'multilingual-e5-base': shape=(47, 768) -> /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados/ejercicio3_pdf_embeddings/embeddings_multilingual-e5-base.npy

Cargando modelo de embeddings: paraphrase-mpnet-base-v2 (sentence-transformers/paraphrase-multilingual-mpnet-base-v2)...


Batches: 100%|██████████| 2/2 [00:03<00:00,  1.55s/it]


✓ Embeddings generados para 'paraphrase-mpnet-base-v2': shape=(47, 768) -> /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados/ejercicio3_pdf_embeddings/embeddings_paraphrase-mpnet-base-v2.npy

Cargando modelo de embeddings: bge-m3 (BAAI/bge-m3)...


Batches: 100%|██████████| 2/2 [00:12<00:00,  6.45s/it]


✓ Embeddings generados para 'bge-m3': shape=(47, 1024) -> /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados/ejercicio3_pdf_embeddings/embeddings_bge-m3.npy

Cargando modelo de embeddings: paraphrase-MiniLM-L12-v2 (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.01s/it]


✓ Embeddings generados para 'paraphrase-MiniLM-L12-v2': shape=(47, 384) -> /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados/ejercicio3_pdf_embeddings/embeddings_paraphrase-MiniLM-L12-v2.npy

--- Modelo: multilingual-e5-base ---
Consulta: ¿Cuál es el principio de precaución en materia ambiental?
Fragmento más similar (similitud=0.7608):
  ��������������� 



������������������������������������������ 

��� 

����������������������������������������� 

��������������� 



�������������������������������������������� ������������������������������������������ �������������������������������������������� ��������������������������������...

--- Modelo: paraphrase-mpnet-base-v2 ---
Consulta: ¿Cuál es el principio de precaución en materia ambiental?
Fragmento más similar (similitud=0.2391):
  ���������������� ������������������������������������������� ������������������������������� 

## ������������������������������� ��������������� 

����������������������������������������

---
## Resumen final

Todos los artefactos (tablas CSV, modelos `.model`, embeddings `.npy`,
gráficas `.png` y conclusiones `.txt`) quedaron guardados dentro de tu
Google Drive, en:

- `Entrega_2_PNL/resultados/ejercicio1_tokenizacion/`
- `Entrega_2_PNL/resultados/ejercicio2_embeddings/`
- `Entrega_2_PNL/resultados/ejercicio3_pdf_embeddings/`

y persistirán entre sesiones de Colab.


In [9]:
import os

for carpeta in (config.OUTPUT_DIR_EJ1, config.OUTPUT_DIR_EJ2, config.OUTPUT_DIR_EJ3):
    print(f"\n{carpeta}:")
    if os.path.exists(carpeta):
        for f in sorted(os.listdir(carpeta)):
            print("  -", f)
    else:
        print("  (vacío)")



/home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados/ejercicio1_tokenizacion:
  - ejercicio1_comparacion_tokenizadores.csv
  - ejercicio1_conclusion.txt

/home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados/ejercicio2_embeddings:
  - comparacion_pca_tsne_FastText_100.txt
  - comparacion_pca_tsne_FastText_10000.txt
  - comparacion_pca_tsne_FastText_5000.txt
  - comparacion_pca_tsne_Word2Vec_100.txt
  - comparacion_pca_tsne_Word2Vec_10000.txt
  - comparacion_pca_tsne_Word2Vec_5000.txt
  - corpus_es_lote_0.npy
  - corpus_es_lote_1.npy
  - corpus_es_lote_2.npy
  - corpus_es_lote_3.npy
  - corpus_es_lote_4.npy
  - ejercicio2_explicacion_oov.txt
  - fasttext_es.model
  - fasttext_es.model.wv.vectors_ngrams.npy
  - pca_FastText_100.png
  - pca_FastText_10000.png
  - pca_FastText_5000.png
  - pca_Word2Vec_100.png
  - pca_Word2Vec_10000.png
  - pca_Word2Vec_5000.png
  - tsne_FastText_100.png
  - tsne_FastText_10000.png
  - tsne_FastText_5000.png
  - tsne_Word2Vec_100.png
  

In [10]:
# Celda de limpieza del proyecto
import shutil
from pathlib import Path

ejecutar_limpieza = True

if ejecutar_limpieza:
    for carpeta in [Path(config.OUTPUT_DIR), Path(config.PDF_DIR)]:
        if carpeta.exists():
            shutil.rmtree(carpeta)
            print(f"Eliminada: {carpeta}")
        else:
            print(f"No existe: {carpeta}")

    for carpeta in [Path(config.OUTPUT_DIR), Path(config.PDF_DIR)]:
        carpeta.mkdir(parents=True, exist_ok=True)

    print("La estructura quedó lista para volver a correr el proyecto.")
else:
    print("La limpieza no se ejecutó.")

Eliminada: /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/resultados
Eliminada: /home/ervin-caravali-ibarra/Documents/Entrega_2_PNL/pdfs_taller
La estructura quedó lista para volver a correr el proyecto.
